# Lesson 6: Behavior

**You will learn:** how to describe a task with `TaskContext`, get answers through a readout, and read implied probabilities and reaction times.
**Requires:** CPU only, under a minute.

A `TaskContext` says what the subject is asked to do; its `response` says what form the answer takes: `Choice(labels)`, `Scale(...)` for ratings, or `Text()`. A network trained for something else answers through a **readout**: a classifier fitted on one layer with labeled examples. Its probabilities and reaction times are **implied** by its activity; a model that replies in words gives **reported** answers instead (see [Experiment 14](../experiments/14_reading_instructions_and_words.ipynb)).

**The question:** is the readout less certain on the images it gets wrong? Its implied reaction time (`evidence_margin`: 1 minus the gap between the two most probable answers) should then be longer on errors.

## Setup

Imports and settings.

In [1]:
# Keep the output short: no progress bars, no library warnings.
import warnings
from functools import partialmethod
from tqdm.auto import tqdm
import tqdm as _tqdm
for _bar in (tqdm, _tqdm.tqdm):
    _bar.__init__ = partialmethod(_bar.__init__, disable=True)
warnings.filterwarnings('ignore')
import os
os.environ['BS_INSTALL_DEPENDENCIES'] = 'no'      # every requirement is already installed

from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image

ASSETS = Path('../assets/lessons')        # weights and images committed with the lessons

# SqueezeNet 1.1 trained on ImageNet (5 MB). These images come from another dataset, CIFAR-10.
net = torchvision.models.squeezenet1_1()
net.load_state_dict(torch.load(ASSETS / 'squeezenet1_1.pth'))
net.eval()                                # inference only: no dropout, no training

# The network expects 224 x 224 images, normalized the way it was trained.
TRANSFORM = torchvision.transforms.Compose([
    torchvision.transforms.Resize(224), torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

def preprocess(paths):
    """Image files -> one (batch, 3, 224, 224) array."""
    return np.stack([TRANSFORM(Image.open(p).convert('RGB')).numpy() for p in paths])

## The subject

A `BrainScoreModel` is assembled from parts. The **extractor** prepares images and records layers; the **region map** says which layer stands for which brain area (a hypothesis, not a fact).

### Letting the subject answer

The network was trained to name 1,000 ImageNet classes, not to say "animal" or "vehicle". To answer this task, the subject uses a **readout**: a small classifier fitted on the activity of one layer, using labelled example images. `behavioral_readout_layer` names that layer.

In [2]:
from brainscore_core import BrainScoreModel
from brainscore.model_helpers.vision_wrapper import VisionWrapper

def squeezenet_subject(network, name):
    return BrainScoreModel(
        identifier=name, model=network,
        extractor={'vision': VisionWrapper(identifier=name, model=network, preprocessing=preprocess)},
        region_layer_map={'V1': 'features.2', 'V4': 'features.5', 'IT': 'features.12'},
        behavioral_readout_layer='features.12')       # answers are read out from this layer

subject = squeezenet_subject(net, 'squeezenet1_1')
sorted(subject.out_channels)                          # what this subject can produce

['behavior',
 'behavior:choice',
 'behavior:probabilities',
 'behavior:rating',
 'behavior:rt',
 'neural:IT',
 'neural:V1',
 'neural:V4']

The null subject is the same architecture with random weights, built the same way.

In [3]:
torch.manual_seed(0)                                           # fixed seed: the same random weights every run
untrained = squeezenet_subject(torchvision.models.squeezenet1_1().eval(), 'squeezenet1_1-untrained')

## The experiment

### Stimuli

In [4]:
from brainscore_core.supported_data_standards.brainio.stimuli import StimulusSet

# 120 natural images, 12 of each CIFAR-10 category, each labelled animal or vehicle.
meta = pd.read_csv(ASSETS / 'images.csv')
stimuli = StimulusSet([{'stimulus_id': row.file[:-4], 'image_path': str(ASSETS / 'images' / row.file),
                        'category': row.category, 'label': row.kind} for row in meta.itertuples()])
stimuli.identifier = 'cifar10-lessons-120'
stimuli.stimulus_paths = dict(zip(stimuli['stimulus_id'], stimuli['image_path']))
stimuli['label'].value_counts().to_dict()

{'animal': 72, 'vehicle': 48}

In [5]:
# Half the images (6 per category) fit the readout; the other half test it.
position = stimuli.groupby('category').cumcount()          # 0..11 within each category
train = StimulusSet(stimuli[position < 6].reset_index(drop=True)); train.identifier = 'lessons-train'
test = StimulusSet(stimuli[position >= 6].reset_index(drop=True)); test.identifier = 'lessons-test'
for part in (train, test):
    part.stimulus_paths = dict(zip(part['stimulus_id'], part['image_path']))
len(train), len(test)

(60, 60)

### Task

The subject answers "animal" or "vehicle" for each test image. In `Choice(...)`, `probabilities=True` also asks for the probability of each answer, and `reaction_time='evidence_margin'` for an implied reaction time: 1 minus the gap between the two most probable answers, so close calls are slow. `fitting_stimuli` are the labelled images the readout learns from, and `prefer_path='readout'` says to answer through the readout even if a subject could also reply in words.

In [6]:
from brainscore_core import TaskContext
from brainscore_core.responses import Choice

task = TaskContext(response=Choice(['animal', 'vehicle'], probabilities=True, reaction_time='evidence_margin'),
                   fitting_stimuli=train, prefer_path='readout')

## Run

`answers()` shows the test images to a subject once and returns one assembly per requested channel. Passing `task_context=task` to the session is what makes the subject answer instead of just viewing.

In [7]:
from brainscore_core.streaming import BufferedSession

def answers(s):
    """Present the test images once; return {'choice', 'probabilities', 'rt'} assemblies."""
    channels = ['behavior:choice', 'behavior:probabilities', 'behavior:rt']
    session = BufferedSession(stimulus_set=test, task_context=task, requested_output_channels=channels)
    s.participate(session)
    s.reset()                                                  # clear the fitted readout
    return {c.split(':')[1]: session.collect(c) for c in channels}   # 'behavior:rt' -> 'rt'

result = answers(subject)
null = answers(untrained)

## Read results back

The choice assembly has one row per test image and one column per label; the chosen label is the column with the 1. The reaction times say in `attrs` how they were obtained.

In [8]:
choice = result['choice']
print(pd.DataFrame(choice.values, index=choice['stimulus_id'].values, columns=choice['choice'].values).head(3))
print('rt measure:', result['rt'].attrs['measure'])

             animal  vehicle
airplane_06     0.0      1.0
airplane_07     0.0      1.0
airplane_08     1.0      0.0
rt measure: {'kind': 'implied', 'link': 'evidence_margin', 'source': 'readout'}


## Analysis

In [9]:
from scipy.stats import mannwhitneyu

LABELS = np.array(['animal', 'vehicle'])
truth = np.asarray(test['label'])

def correct(choices):
    """True for each test image whose chosen label is the true one."""
    chosen = LABELS[np.asarray(choices.values).argmax(axis=1)]   # the column with the 1
    return chosen == truth

def errors_slower(rt, is_correct):
    """One-sided Mann-Whitney p-value that reaction times are longer on errors."""
    return mannwhitneyu(rt[~is_correct], rt[is_correct], alternative='greater').pvalue

## Results

In [10]:
is_correct = correct(result['choice'])
null_correct = correct(null['choice'])
rt = np.asarray(result['rt'].values)
print('held-out accuracy:', round(is_correct.mean(), 3))
print('always answering "animal":', round(np.mean(truth == 'animal'), 3))   # 36 of the 60 test images are animals
print('untrained network, held-out accuracy:', round(null_correct.mean(), 3))
print(f'implied RT: correct {rt[is_correct].mean():.3f} (n={is_correct.sum()}), '
      f'errors {rt[~is_correct].mean():.3f} (n={(~is_correct).sum()})')
if (~is_correct).any():
    print('errors slower: p =', round(errors_slower(rt, is_correct), 4))

held-out accuracy: 0.833
always answering "animal": 0.6
untrained network, held-out accuracy: 0.767
implied RT: correct 0.172 (n=50), errors 0.364 (n=10)
errors slower: p = 0.3717


Chance is not 0.5. Six of the ten CIFAR-10 categories are animals, so answering "animal" to every image scores 0.600. The trained readout's 0.833 is measured against that, and the untrained network's 0.767 shows how much a readout gets from features that were never trained.

Read the p-value with the number of errors beside it: with ten or so errors on 60 test images, even a large difference in means may not be reliable, and the answer is to test more images, not to trust the means. A readout that is reliably slower on its errors behaves like a participant whose hard trials are the ones they get wrong; a reaction-time benchmark would test whether its slow images are people's slow images.

Implied reaction times come from the readout's confidence, so they mean something only on images like the ones it was fitted on. On noisy or unfamiliar images a readout can become *more* confident in one answer and, by this measure, faster as the task gets harder: the opposite of people. Check that a reaction-time effect holds on in-distribution images before reading it as behavior.

## Summary

- `TaskContext(response=Choice(...), fitting_stimuli=...)` describes the task; held-out stimuli test it.
- Readouts give implied probabilities and reaction times, labelled in `attrs['measure']`.
- Compare accuracy with the base rate (here 0.600, always "animal") and with an untrained network.

**Next:** [Lesson 7](07_perturb_a_model.ipynb) changes the network during a run.